# Module 5: reproducibility, SOTA, explainability, and ablations

This notebook uses `core.module05_fusion_classifier` and retains the SOTA audit/benchmark/comparison flow from [01_train_val_test_batch_runner.ipynb](./01_train_val_test_batch_runner.ipynb).

The architecture matrix uses the nine existing Module 5 experiments, `center` pooling, NTv1 DNA length 1001, ESM2 protein length 501, all 28 requested tabular features (11 AF/conservation + 9 SpliceAI + 8 k=64 geometry), three separate training splits (`train1`, `train2`, `train3`), and the four test datasets. The modality matrix additionally tests all 3 DNA families × 4 protein families at DNA=1001/protein=501. A checkpoint is fitted once per train/validation configuration and reused for its four test datasets.

**Prerequisites observed in this workspace:** `fm_embeddings/sequence_ablation` currently has the requested 1001/501 embeddings for train3/val/test, but not train1/train2. The legacy train1/train2 embeddings are 601/101 and must not be substituted. `geometry_v2_k64` currently has only the `train3` normalization context. Generate the missing model-length embeddings and k=64 geometry for train1/train2 before enabling full-suite stages. The preflight below fails explicitly if any required artifact is missing.

Set the stage flags in the final cell deliberately; these matrices can take substantial compute. The sequence-length stage tests the full Cartesian product of four DNA lengths and five protein lengths (20 pairs), including the 1001/501 reference, which is reused from the architecture stage. NTv1 length 10001 is excluded because the existing tokenizer support manifest marks it unsupported.

In [1]:
from datetime import datetime
from pathlib import Path
import json
import os
import sys

import pandas as pd
import torch
from IPython.display import display

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents)
     if (candidate / "controllers").is_dir() and (candidate / "experiments").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repository root from {_cwd}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.module05_fusion_classifier.dataset import (
    BIO_CORE_COLUMNS,
    SPLICEAI_COLUMNS,
)
from core.module05_fusion_classifier.pipeline import FusionBatchPipeline
from core.module05_fusion_classifier.sota_benchmark import (
    evaluate_sota_from_test_files,
    SOTA_MODEL_COLUMNS_FULL,
    SOTA_REQUIRED_COLUMNS,
)
from core.module05_fusion_classifier import pipeline as pipeline_module

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR = r"D:\variant_data"
EMBEDDING_DIR = os.path.join(BASE_DIR, "fm_embeddings", "sequence_ablation")
FM_PROFILE_JSON = os.path.join(BASE_DIR, "profiling", "fm_profiling_sequence_ablation.json")
GEOMETRY_K = 64
GEOMETRY_DIR = os.path.join(BASE_DIR, f"geometry_v2_k{GEOMETRY_K}", "{context}")
GEOM_FEATURE_NAMES = [
    f"{modality}_{feature}"
    for modality in ("dna", "prot")
    for feature in ("LID", "LVD_L2", "LVD_Relative", "LVD_Cosine")
]
POOLINGS = ["center", "cls", "mean"]
DNA_LENGTHS = [301, 601, 1001, 2001]
PROTEIN_LENGTHS = [31, 101, 201, 501, 1001]
DNA_MODEL = "nt_v1_500m_dna_1001"
PROTEIN_MODEL = "esm2_650m_prot_501"
REFERENCE_POOLING = "center"

MODELS_SPACE = [
    {"name": f"nt_v1_500m_dna_{length}", "seq_type": "dna", "sequence_length": length}
    for length in DNA_LENGTHS
] + [
    {"name": f"esm2_650m_prot_{length}", "seq_type": "protein", "sequence_length": length}
    for length in PROTEIN_LENGTHS
] + [
    {"name": f"{family}_dna_1001", "seq_type": "dna", "sequence_length": 1001}
    for family in ("nt_v3_650m", "nt_v2_500m")
] + [
    {"name": f"{family}_prot_501", "seq_type": "protein", "sequence_length": 501}
    for family in ("esm1b_650m", "esm1v_650m", "esmc_600m")
]
PAIR_DNA_MODELS = [
    "nt_v1_500m_dna_1001",
    "nt_v3_650m_dna_1001",
    "nt_v2_500m_dna_1001",
]
PAIR_PROTEIN_MODELS = [
    "esm1b_650m_prot_501",
    "esm1v_650m_prot_501",
    "esm2_650m_prot_501",
    "esmc_600m_prot_501",
]
EXPERIMENTS = [
    {"name": "PyTorch_Concat", "type": "pytorch", "fusion": "concat"},
    {"name": "PyTorch_CrossAttn", "type": "pytorch", "fusion": "cross_attention"},
    {"name": "PyTorch_Transformer", "type": "pytorch", "fusion": "transformer"},
    {"name": "PyTorch_Gating", "type": "pytorch", "fusion": "gating"},
    {"name": "Pure_XGBoost_Concat", "type": "xgboost_pure", "fusion": "concat"},
    {"name": "Hybrid_Concat_XGBoost", "type": "hybrid", "fusion": "concat"},
    {"name": "Hybrid_CrossAttn_XGBoost", "type": "hybrid", "fusion": "cross_attention"},
    {"name": "Hybrid_Transformer_XGBoost", "type": "hybrid", "fusion": "transformer"},
    {"name": "Hybrid_Gating_XGBoost", "type": "hybrid", "fusion": "gating"},
]
TEST_SPLITS = {
    "test": "Test",
    "clinvarhq": "ClinVarHQ",
    "uniprot": "UniProt",
    "proteingym": "ProteinGym",
}
TRAIN_SPLITS = ["train1", "train2", "train3"]
DATASETS = [
    {
        "name": f"{train.title()}Val_{test_label}",
        "train": train,
        "val": "val",
        "test": test_split,
    }
    for train in TRAIN_SPLITS
    for test_split, test_label in TEST_SPLITS.items()
]
CONFIG = {
    "batch_size": 256,
    "epochs": 30,
    "lr": 1e-4,
    "weight_decay": 1e-3,
    "lr_factor": 0.5,
    "lr_patience": 3,
    "early_stop_patience": 6,
    "num_workers": 0,
    "seed": 2,
    "xgb_n_jobs": 1,
}
EXPLAINABILITY = {
    "enable_shap": True,
    "enable_lime": True,
    "max_background_samples": 512,
    "max_explain_samples": 200,
    "lime_num_features": 20,
    "max_lime_samples": 20,
    "lime_num_samples": 2000,
    "shap_top_k": 30,
    "random_state": 2,
}

RESUME_EXISTING_RUN = False
RESUME_BATCH_RUN_DIR = None
if RESUME_EXISTING_RUN:
    if RESUME_BATCH_RUN_DIR is None:
        raise ValueError("Set RESUME_BATCH_RUN_DIR when RESUME_EXISTING_RUN=True")
    RUN_OUTPUT_DIR = Path(RESUME_BATCH_RUN_DIR).resolve()
else:
    _stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    RUN_OUTPUT_DIR = REPO_ROOT / "experiments" / f"reproduction_ablations_{_stamp}"
    _suffix = 2
    while RUN_OUTPUT_DIR.exists():
        RUN_OUTPUT_DIR = REPO_ROOT / "experiments" / f"reproduction_ablations_{_stamp}_{_suffix:02d}"
        _suffix += 1

pipeline = FusionBatchPipeline(
    base_dir=BASE_DIR,
    config=CONFIG,
    datasets=DATASETS,
    models_space=MODELS_SPACE,
    pooling_strategies=POOLINGS,
    experiments=EXPERIMENTS,
    explainability=EXPLAINABILITY,
    fm_profile_json=FM_PROFILE_JSON,
    geometry_dir=GEOMETRY_DIR,
    geom_feature_names=GEOM_FEATURE_NAMES,
    batch_run_dir=str(RUN_OUTPUT_DIR),
    embedding_dir=EMBEDDING_DIR,
)
print(f"[*] Device: {DEVICE} | batch: {pipeline.batch_run_dir}")
print(f"[*] Datasets: {len(DATASETS)} train/test combinations | models: {len(MODELS_SPACE)}")


c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Device: cuda | batch: D:\Missense_variant_pathogenicity_prediction\experiments\reproduction_ablations_20261101_111011
[*] Datasets: 12 train/test combinations | models: 14


## SOTA benchmark and artifact preflight

The SOTA definitions, strict required-column audit, thresholds, label candidates, and four source files mirror Notebook 01. This cell can be run before training; it writes the SOTA metrics, mapping, and audit CSVs into the current batch directory.

In [2]:
SOTA_TEST_FILES = {
    "test": os.path.join(BASE_DIR, "test_full_seq_after_vep_final.parquet"),
    "clinvarhq": os.path.join(BASE_DIR, "clinvarhq_full_seq_after_vep_final.parquet"),
    "uniprot": os.path.join(BASE_DIR, "uniprot_full_seq_after_vep_final.parquet"),
    "proteingym": os.path.join(BASE_DIR, "proteingym_full_seq_after_vep_final.parquet"),
}
SOTA_LABEL_COL = None
SOTA_LABEL_CANDIDATES = ["Pathogenicity_Label", "Label"]
SOTA_THRESHOLD = 0.5
SOTA_ENFORCE_REQUIRED = True

sota_result = evaluate_sota_from_test_files(
    test_files=SOTA_TEST_FILES,
    output_dir=pipeline.batch_run_dir,
    label_col=SOTA_LABEL_COL,
    label_candidates=SOTA_LABEL_CANDIDATES,
    model_columns=SOTA_MODEL_COLUMNS_FULL,
    threshold=SOTA_THRESHOLD,
    required_columns=SOTA_REQUIRED_COLUMNS,
    enforce_required_columns=SOTA_ENFORCE_REQUIRED,
)
print(f"[SOTA] metrics={sota_result['metrics_path']} | rows={sota_result['num_rows']}")
print(f"[SOTA] mapping={sota_result['mapping_path']} | models={sota_result['num_models']}")
if sota_result.get("audit") is not None:
    audit = sota_result["audit"]
    print(f"[SOTA] audit={audit['audit_path']} | missing={audit['missing_count']} | null={audit['null_issue_count']}")

BIO_FEATURE_GROUPS = {
    "allele_frequencies": ["AF", "gnomADe_AF"],
    "conservation_scores": [column for column in BIO_CORE_COLUMNS if column not in {"AF", "gnomADe_AF"}],
    "spliceai": list(SPLICEAI_COLUMNS),
    "geometry": list(GEOM_FEATURE_NAMES),
    "dna_geometry": [name for name in GEOM_FEATURE_NAMES if name.startswith("dna_")],
    "protein_geometry": [name for name in GEOM_FEATURE_NAMES if name.startswith("prot_")],
}
BIO_FEATURES = [*BIO_CORE_COLUMNS, *SPLICEAI_COLUMNS]
ALL_TABULAR_FEATURES = [*BIO_FEATURES, *GEOM_FEATURE_NAMES]
if len(ALL_TABULAR_FEATURES) != 28 or len(set(ALL_TABULAR_FEATURES)) != 28:
    raise RuntimeError(f"Expected 28 unique AF/conservation/SpliceAI/geometry features, got {len(ALL_TABULAR_FEATURES)}")

def full_feature_config(label="bio_core_dna_prot_spliceai_geom", **overrides):
    config = {
        "label": label,
        "active_modalities": ["dna", "prot", "bio_core", "spliceai", "geom"],
        "dna_model_name": DNA_MODEL,
        "prot_model_name": PROTEIN_MODEL,
        "geom_feature_names": list(GEOM_FEATURE_NAMES),
        "bio_feature_names": list(BIO_FEATURES),
        "poolings": [REFERENCE_POOLING],
        "experiment_names": [experiment["name"] for experiment in EXPERIMENTS],
    }
    config.update(overrides)
    return config

def preflight_configs(configs):
    missing = set()
    if not os.path.exists(FM_PROFILE_JSON):
        missing.add(f"Missing foundation-model profiling JSON: {FM_PROFILE_JSON}")
        fm_profile = {}
    else:
        with open(FM_PROFILE_JSON, "r", encoding="utf-8") as handle:
            fm_profile = json.load(handle)
    missing_profile = set()
    for config in configs:
        modalities = {str(mod).lower() for mod in config["active_modalities"]}
        dna_name = config.get("dna_model_name")
        prot_name = config.get("prot_model_name")
        for modality, model_name in (("dna", dna_name), ("prot", prot_name)):
            if model_name is None or modality not in modalities:
                continue
            for split in TEST_SPLITS:
                stats = fm_profile.get(model_name, {}).get(split, {})
                if float(stats.get("inference_time_ms", 0.0) or 0.0) <= 0:
                    missing_profile.add(f"{model_name}:{split}")
        for dataset in DATASETS:
            context = dataset["train"]
            for split in (dataset["train"], dataset["val"], dataset["test"]):
                try:
                    pipeline_module._resolve_bio_path(pipeline.bio_dir, split)
                except FileNotFoundError as exc:
                    missing.add(str(exc))
                for pooling in config.get("poolings", [REFERENCE_POOLING]):
                    for modality, model_name in (("dna", dna_name), ("prot", prot_name)):
                        if model_name is None:
                            continue
                        if modality in modalities:
                            try:
                                pipeline_module._resolve_pt_path(pipeline.embed_dir, split, model_name, pooling)
                            except FileNotFoundError as exc:
                                missing.add(str(exc))
                        if "geom" in modalities and any(name.startswith(f"{modality}_") for name in config["geom_feature_names"]):
                            try:
                                geometry_root = GEOMETRY_DIR.format(context=context)
                                pipeline_module._resolve_geom_path(geometry_root, split, model_name, pooling)
                            except FileNotFoundError as exc:
                                missing.add(str(exc))
    if missing_profile:
        missing.add("Missing FM inference profiling records: " + ", ".join(sorted(missing_profile)))
    if missing:
        examples = "\n".join(sorted(missing)[:30])
        raise FileNotFoundError(
            f"Preflight found {len(missing)} missing input artifacts. Prepare the required embeddings/geometry before running this stage.\n{examples}"
        )
    print(f"[PREFLIGHT PASS] {len(configs)} configurations across {len(DATASETS)} train/test combinations.")

print(f"[*] Feature groups: { {name: len(columns) for name, columns in BIO_FEATURE_GROUPS.items()} }")
print(f"[*] Total tabular features: {len(ALL_TABULAR_FEATURES)}")


[SOTA] metrics=D:\Missense_variant_pathogenicity_prediction\experiments\reproduction_ablations_20261101_111011/sota_metrics.csv | rows=112
[SOTA] mapping=D:\Missense_variant_pathogenicity_prediction\experiments\reproduction_ablations_20261101_111011/sota_column_mapping.csv | models=28
[SOTA] audit=D:\Missense_variant_pathogenicity_prediction\experiments\reproduction_ablations_20261101_111011/sota_required_columns_audit.csv | missing=0 | null=0
[*] Feature groups: {'allele_frequencies': 2, 'conservation_scores': 9, 'spliceai': 9, 'geometry': 8, 'dna_geometry': 4, 'protein_geometry': 4}
[*] Total tabular features: 28


## Study matrix

The feature-group and individual-feature studies are leave-one-group/feature-out from the 28-feature reference. Sequence length varies one modality at a time while the other stays at 1001/501. Pooling, sequence, modality, and feature-ablation stages use pure XGBoost unless edited.

In [3]:
XGB_ONLY = ["Pure_XGBoost_Concat"]

def build_architecture_configs():
    return [full_feature_config()]

def build_explainability_configs():
    return [full_feature_config(
        label="bio_core_dna_prot_spliceai_geom_shap_lime",
        run_explainability=True,
        experiment_names=XGB_ONLY,
    )]

def build_group_ablation_configs():
    configs = []
    for group_name, removed in BIO_FEATURE_GROUPS.items():
        bio_features = [name for name in BIO_FEATURES if name not in removed]
        geom_features = [name for name in GEOM_FEATURE_NAMES if name not in removed]
        active = ["dna", "prot", "bio_core", "spliceai", "geom"]
        if not any(name in bio_features for name in BIO_CORE_COLUMNS):
            active.remove("bio_core")
        if not any(name in bio_features for name in SPLICEAI_COLUMNS):
            active.remove("spliceai")
        if not geom_features:
            active.remove("geom")
        configs.append(full_feature_config(
            label=f"bio_drop_group_{group_name}",
            active_modalities=active,
            bio_feature_names=bio_features or None,
            geom_feature_names=geom_features or None,
            experiment_names=XGB_ONLY,
        ))
    return configs

def build_single_feature_ablation_configs():
    configs = []
    for removed in ALL_TABULAR_FEATURES:
        bio_features = [name for name in BIO_FEATURES if name != removed]
        geom_features = [name for name in GEOM_FEATURE_NAMES if name != removed]
        configs.append(full_feature_config(
            label=f"bio_drop_feature_{removed}",
            bio_feature_names=bio_features,
            geom_feature_names=geom_features,
            experiment_names=XGB_ONLY,
        ))
    return configs

def build_pooling_configs():
    config = full_feature_config(experiment_names=XGB_ONLY)
    config["poolings"] = list(POOLINGS)
    return [config]

def build_sequence_length_configs():
    configs = []
    for dna_length in DNA_LENGTHS:
        for protein_length in PROTEIN_LENGTHS:
            is_reference = dna_length == 1001 and protein_length == 501
            label = (
                "bio_core_dna_prot_spliceai_geom"
                if is_reference
                else f"sequence_pair_dna_{dna_length}_protein_{protein_length}"
            )
            configs.append(full_feature_config(
                label=label,
                dna_model_name=f"nt_v1_500m_dna_{dna_length}",
                prot_model_name=f"esm2_650m_prot_{protein_length}",
                experiment_names=XGB_ONLY,
            ))
    return configs

def build_modality_configs():
    configs = [
        full_feature_config(label="bio_core_dna_prot_spliceai_geom", experiment_names=XGB_ONLY),
        full_feature_config(
            label="modalities_dna_only_bio_splice_geom",
            active_modalities=["dna", "bio_core", "spliceai", "geom"],
            experiment_names=XGB_ONLY,
        ),
        full_feature_config(
            label="modalities_protein_only_bio_splice_geom",
            active_modalities=["prot", "bio_core", "spliceai", "geom"],
            experiment_names=XGB_ONLY,
        ),
        full_feature_config(
            label="modalities_bio_splice_geom_only",
            active_modalities=["bio_core", "spliceai", "geom"],
            experiment_names=XGB_ONLY,
        ),
        full_feature_config(
            label="modalities_dna_only",
            active_modalities=["dna"],
            bio_feature_names=None,
            geom_feature_names=None,
            experiment_names=XGB_ONLY,
        ),
        full_feature_config(
            label="modalities_protein_only",
            active_modalities=["prot"],
            bio_feature_names=None,
            geom_feature_names=None,
            experiment_names=XGB_ONLY,
        ),
        full_feature_config(
            label="modalities_dna_protein_only",
            active_modalities=["dna", "prot"],
            bio_feature_names=None,
            geom_feature_names=None,
            experiment_names=XGB_ONLY,
        ),
        full_feature_config(
            label="modalities_bio_only",
            active_modalities=["bio_core"],
            dna_model_name=None,
            prot_model_name=None,
            bio_feature_names=list(BIO_CORE_COLUMNS),
            geom_feature_names=None,
            experiment_names=XGB_ONLY,
        ),
        full_feature_config(
            label="modalities_spliceai_only",
            active_modalities=["spliceai"],
            dna_model_name=None,
            prot_model_name=None,
            bio_feature_names=list(SPLICEAI_COLUMNS),
            geom_feature_names=None,
            experiment_names=XGB_ONLY,
        ),
        full_feature_config(
            label="modalities_geometry_only",
            active_modalities=["geom"],
            bio_feature_names=None,
            experiment_names=XGB_ONLY,
        ),
    ]
    for dna_name in PAIR_DNA_MODELS:
        for prot_name in PAIR_PROTEIN_MODELS:
            if dna_name == DNA_MODEL and prot_name == PROTEIN_MODEL:
                continue
            dna_family = dna_name.removesuffix("_dna_1001")
            protein_family = prot_name.removesuffix("_prot_501")
            configs.append(full_feature_config(
                label=f"model_pair_{dna_family}_{protein_family}_bio_splice_geom",
                dna_model_name=dna_name,
                prot_model_name=prot_name,
                experiment_names=XGB_ONLY,
            ))
    return configs

def run_configs(configs, resume=True):
    preflight_configs(configs)
    return pipeline.run_config_matrix(configs, resume=resume)


## Run selected stages

Enable only the stages to run in this session. Architecture comparison includes all nine existing fusion/classifier architectures. Set `RUN_EXPLAINABILITY` separately to fit the pure-XGBoost reference with both SHAP and LIME enabled.

In [4]:
RUN_ARCHITECTURE = True
RUN_EXPLAINABILITY = True
RUN_BIOLOGICAL_GROUP_ABLATION = True
RUN_28_FEATURE_ABLATION = True
RUN_POOLING_ABLATION = True
RUN_SEQUENCE_LENGTH_ABLATION = True
RUN_DNA_PROTEIN_ABLATION = True
RESUME = True

STAGES = [
    (RUN_ARCHITECTURE, "9-architecture comparison", build_architecture_configs),
    (RUN_EXPLAINABILITY, "SHAP + LIME", build_explainability_configs),
    (RUN_BIOLOGICAL_GROUP_ABLATION, "biological feature-group ablation", build_group_ablation_configs),
    (RUN_28_FEATURE_ABLATION, "28 individual-feature ablation", build_single_feature_ablation_configs),
    (RUN_POOLING_ABLATION, "pooling ablation", build_pooling_configs),
    (RUN_SEQUENCE_LENGTH_ABLATION, "sequence-length ablation", build_sequence_length_configs),
    (RUN_DNA_PROTEIN_ABLATION, "DNA/protein modality ablation", build_modality_configs),
]
selected_stages = [(name, builder()) for enabled, name, builder in STAGES if enabled]
if not selected_stages:
    print("[SKIP] No training stage enabled. Set one or more RUN_* flags to True.")
else:
    for stage_name, configs in selected_stages:
        print(f"\n{'=' * 80}\n[STAGE] {stage_name} | configs={len(configs)}\n{'=' * 80}")
        result = run_configs(configs, resume=RESUME)
        print(f"[DONE] {stage_name} | metrics={result['metrics_path']} | metric_rows={result['num_rows_metrics']} | fit_profile={result['fit_profiling_path']} | fit_rows={result['num_rows_fit_profiling']}")

print(f"[OUTPUT] {pipeline.batch_run_dir}")



[STAGE] 9-architecture comparison | configs=1
[PREFLIGHT PASS] 1 configurations across 12 train/test combinations.

[CONFIG MATRIX] bio_core_dna_prot_spliceai_geom | modalities=['dna', 'prot', 'bio_core', 'spliceai', 'geom'] | poolings=['center'] | architectures=9
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 104385 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng

[Epoch 01] train_loss=0.2118 | val_loss=0.1556 | val_mcc=0.8082 | val_f1=0.8399 | val_auroc=0.9758


[Epoch 02] train_loss=0.1615 | val_loss=0.1509 | val_mcc=0.8168 | val_f1=0.8498 | val_auroc=0.9761


[Epoch 03] train_loss=0.1422 | val_loss=0.1532 | val_mcc=0.8153 | val_f1=0.8480 | val_auroc=0.9769


[Epoch 04] train_loss=0.1234 | val_loss=0.1579 | val_mcc=0.8170 | val_f1=0.8494 | val_auroc=0.9762


...

[Test] train1__val_center_model_pair_nt_v2_500m_esmc_600m_bio_splice_geom_nt_v2_500m_dna_1001_esmc_600m_prot_501_Pure_XGBoost_Concat__geom_dna_LID+dna_LVD_L2+dna_LVD_Relative+dna_LVD_Cosine+prot_LID+prot_LVD_L2+prot_LVD_Relative+prot_LVD_Cosine | test=Train1Val_Test | MCC=0.8439 | F1=0.8862 | AUROC=0.9786 | AUPRC=0.9510
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 104385 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp 

D:\Missense_variant_pathogenicity_prediction\core\module05_fusion_classifier\pipeline.py:338: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_fit = pd.concat([df_fit, pd.DataFrame([row])], ignore_index=True)


[Test] train2__val_center_model_pair_nt_v2_500m_esmc_600m_bio_splice_geom_nt_v2_500m_dna_1001_esmc_600m_prot_501_Pure_XGBoost_Concat__geom_dna_LID+dna_LVD_L2+dna_LVD_Relative+dna_LVD_Cosine+prot_LID+prot_LVD_L2+prot_LVD_Relative+prot_LVD_Cosine | test=Train2Val_Test | MCC=0.8354 | F1=0.8789 | AUROC=0.9769 | AUPRC=0.9492
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 67990 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp v

D:\Missense_variant_pathogenicity_prediction\core\module05_fusion_classifier\pipeline.py:338: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_fit = pd.concat([df_fit, pd.DataFrame([row])], ignore_index=True)


[Test] train3__val_center_model_pair_nt_v2_500m_esmc_600m_bio_splice_geom_nt_v2_500m_dna_1001_esmc_600m_prot_501_Pure_XGBoost_Concat__geom_dna_LID+dna_LVD_L2+dna_LVD_Relative+dna_LVD_Cosine+prot_LID+prot_LVD_L2+prot_LVD_Relative+prot_LVD_Cosine | test=Train3Val_Test | MCC=0.8323 | F1=0.8780 | AUROC=0.9734 | AUPRC=0.9417
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp v

## Expected output files

All outputs are written under `pipeline.batch_run_dir` (`experiments/reproduction_ablations_<timestamp>` by default):

- `sota_metrics.csv`, `sota_column_mapping.csv`, `sota_metric_audit.csv`, and `sota_required_columns_audit.csv` for the four SOTA test sets.
- `global_leaderboard_metrics.csv` with test and validation metrics for every completed OURS row; `global_leaderboard_profiling.csv` with per-test inference/profile measurements; `global_leaderboard_fit_profiling.csv` with one row per unique train/validation fit, linked to the other tables by `Fit_ID`.
- Fit profiling records training duration, train/validation sample counts, epochs, batches and optimizer steps where applicable, effective train-sample throughput, device/GPU, and peak GPU memory. If an existing checkpoint has no recorded training profile, the row explicitly marks its training time as unavailable rather than fabricating a measurement.
- Per-config checkpoint/cache folders, validation-selected `decision_threshold.json`, and `tests/<dataset>/test_probabilities.csv` with Variant_ID, true label, predicted probability, threshold, and predicted class.
- Explainability runs add `tests/<dataset>/explainability/shap_global_importance.csv`, `shap_global_importance_by_true_class.csv`, `shap_local_top_ranked.csv`, `shap_run_info.json`, `lime_global_importance.csv`, `lime_local_explanations.csv`, and `lime_run_info.json` (subject to SHAP/LIME packages being installed).
- `global_compare_ours_vs_sota.csv` after at least one training stage has written metrics.

| Stage | Unique fits | OURS metric rows |
|---|---:|---:|
| 9 architecture | 27 | 108 |
| XGBoost SHAP/LIME | 3 | 12 |
| 6 feature-group ablations | 18 | 72 |
| 28 individual-feature ablations | 84 | 336 |
| Pooling (`center` reused; `cls` and `mean` new) | 6 | 24 |
| Sequence-length ablation (20 pairs; reference reused) | 57 | 228 |
| Modality and 3×4 DNA/protein model-pair ablations | 60 | 240 |
| **Total** | **255** | **1,020** |

If all seven training stages complete in one batch, expected metric and inference-profile rows are 1,020 OURS plus 112 SOTA rows (1,132 in the comparison); expected unique fit-profile rows are 255. The center-pooling XGBoost baseline and the reference DNA+protein sequence pair are reused from the architecture stage rather than fitted twice. These are expected counts, not a promise: missing artifacts or skipped/failed stages reduce the rows.

## Compare OURS vs SOTA

Run this after the selected training stages. The SOTA rows follow Notebook 01's score/rankscore/pred mapping; `Train_Split` differentiates models trained on train1, train2, and train3.

In [5]:
if not Path(pipeline.metrics_csv_path).exists():
    print("[COMPARE SKIP] No OURS metrics yet. Enable and run at least one RUN_* training stage first.")
else:
    ours = pd.read_csv(pipeline.metrics_csv_path)
    if ours.empty:
        print("[COMPARE SKIP] OURS metrics file is empty. Run at least one training stage first.")
    else:
        sota = pd.read_csv(sota_result["metrics_path"])
        test_name_map = {
            f"{train.title()}Val_{display_name}": test_split
            for train in TRAIN_SPLITS
            for test_split, display_name in TEST_SPLITS.items()
        }
        ours["Source"] = "OURS"
        ours["Train_Split"] = ours["Dataset"].astype(str).str.extract(r"^(Train[123])Val_", expand=False)
        ours["Test_Split"] = ours["Dataset"].map(test_name_map)
        sota["Source"] = "SOTA"
        sota["Train_Split"] = "N/A"
        sota["Test_Split"] = sota["Dataset"]
        comparison = pd.concat([ours, sota], ignore_index=True, sort=False)
        comparison = comparison.sort_values(["Test_Split", "Train_Split", "MCC"], ascending=[True, True, False])
        compare_path = Path(pipeline.batch_run_dir) / "global_compare_ours_vs_sota.csv"
        comparison.to_csv(compare_path, index=False)
        print(f"[COMPARE] {compare_path} | rows={len(comparison)}")
        display(comparison[["Test_Split", "Train_Split", "Source", "Ablation", "Network", "MCC", "AUROC", "AUPRC"]].head(60))
        display(pd.read_csv(sota_result["mapping_path"]).head(40))
        if sota_result.get("audit") is not None:
            audit = pd.read_csv(sota_result["audit"]["audit_path"])
            display(audit.query("exists == False or null_count > 0").head(100))


[COMPARE] D:\Missense_variant_pathogenicity_prediction\experiments\reproduction_ablations_20261101_111011\global_compare_ours_vs_sota.csv | rows=1132


,Test_Split,Train_Split,Source,Ablation,Network,MCC,AUROC,AUPRC
1066,clinvarhq,N/A,SOTA,sota,SOTA_ClinPred,0.9268,0.9912,0.9866
1055,clinvarhq,N/A,SOTA,sota,SOTA_MetaRNN,0.8957,0.9928,0.9921
1052,clinvarhq,N/A,SOTA,sota,SOTA_MutationTaster,0.8937,0.9867,0.9874
1064,clinvarhq,N/A,SOTA,sota,SOTA_BayesDel_addAF,0.8890,0.9903,0.9900
1057,clinvarhq,N/A,SOTA,sota,SOTA_REVEL,0.8275,0.9770,0.9738
1070,clinvarhq,N/A,SOTA,sota,SOTA_AlphaMissense,0.8216,0.9613,0.9517
1071,clinvarhq,N/A,SOTA,sota,SOTA_PHACTboost,0.8197,0.9770,0.9747
1068,clinvarhq,N/A,SOTA,sota,SOTA_VARITY_R,0.8052,0.9600,0.9563
1069,clinvarhq,N/A,SOTA,sota,SOTA_VARITY_ER,0.7705,0.9503,0.9445
1065,clinvarhq,N/A,SOTA,sota,SOTA_BayesDel_noAF,0.7617,0.9660,0.9648


,Dataset,Model,file_path,score_col,rankscore_col,pred_col,resolved_prob_col,resolved_prob_kind,resolved_threshold,pred_source,...,binary_metric_source,pred_known_count,pred_unknown_count,pred_positive_count,threshold_positive_count,pred_threshold_disagreement_count,module1_rule_score_col,module1_rule_threshold,module1_rule_checked_count,module1_rule_mismatch_count
0,test,SIFT,D:\variant_data\test_full_seq_after_vep_final....,['SIFT_score'],"['SIFT_converted_rankscore', 'SIFT_rankscore']",['SIFT_pred'],SIFT_converted_rankscore,rankscore,0.39575,SIFT_pred,...,pred,6095,0,2418,2521,103,NaN,NaN,0,0
1,test,SIFT4G,D:\variant_data\test_full_seq_after_vep_final....,['SIFT4G_score'],"['SIFT4G_converted_rankscore', 'SIFT4G_ranksco...",['SIFT4G_pred'],SIFT4G_converted_rankscore,rankscore,0.39575,SIFT4G_pred,...,pred,6095,0,2022,2502,480,NaN,NaN,0,0
2,test,Polyphen2_HDIV,D:\variant_data\test_full_seq_after_vep_final....,['Polyphen2_HDIV_score'],['Polyphen2_HDIV_rankscore'],['Polyphen2_HDIV_pred'],Polyphen2_HDIV_rankscore,rankscore,0.38028,Polyphen2_HDIV_pred,...,pred,6095,0,2468,2497,131,NaN,NaN,0,0
3,test,Polyphen2_HVAR,D:\variant_data\test_full_seq_after_vep_final....,['Polyphen2_HVAR_score'],['Polyphen2_HVAR_rankscore'],['Polyphen2_HVAR_pred'],Polyphen2_HVAR_rankscore,rankscore,0.48762,Polyphen2_HVAR_pred,...,pred,6095,0,2139,2144,137,NaN,NaN,0,0
4,test,MutationTaster,D:\variant_data\test_full_seq_after_vep_final....,['MutationTaster_score'],['MutationTaster_rankscore'],['MutationTaster_pred'],MutationTaster_rankscore,rankscore,0.31733,MutationTaster_pred,...,pred,6095,0,1776,2936,1180,NaN,NaN,0,0
5,test,MetaSVM,D:\variant_data\test_full_seq_after_vep_final....,['MetaSVM_score'],['MetaSVM_rankscore'],['MetaSVM_pred'],MetaSVM_rankscore,rankscore,0.82257,MetaSVM_pred,...,pred,6095,0,1680,1677,3,NaN,NaN,0,0
6,test,MetaLR,D:\variant_data\test_full_seq_after_vep_final....,['MetaLR_score'],['MetaLR_rankscore'],['MetaLR_pred'],MetaLR_rankscore,rankscore,0.81101,MetaLR_pred,...,pred,6095,0,1777,1776,1,NaN,NaN,0,0
7,test,MetaRNN,D:\variant_data\test_full_seq_after_vep_final....,['MetaRNN_score'],['MetaRNN_rankscore'],['MetaRNN_pred'],MetaRNN_rankscore,rankscore,0.61490,MetaRNN_pred,...,pred,6095,0,2003,1721,282,NaN,NaN,0,0
8,test,M-CAP,D:\variant_data\test_full_seq_after_vep_final....,['M-CAP_score'],['M-CAP_rankscore'],['M-CAP_pred'],M-CAP_rankscore,rankscore,0.50000,M-CAP_pred,...,pred,6095,0,2976,2870,106,NaN,NaN,0,0
9,test,REVEL,D:\variant_data\test_full_seq_after_vep_final....,['REVEL_score'],['REVEL_rankscore'],['REVEL_pred'],REVEL_rankscore,rankscore,0.50000,REVEL_pred,...,pred,6095,0,1679,2582,903,REVEL_score,0.5000,6095,0


,Dataset,file_path,column,exists,null_count
